<a href="https://colab.research.google.com/github/irajamuller/error_corrections/blob/main/UA3_UA4_Estabilizadores_Bit_flip_Ruido_Medida.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install qiskit qiskit_aer pylatexenc --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 3.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.3/9.3 MB 60.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 64.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 51.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.6/54.6 kB 3.0 MB/s eta 0:00:00


In [ ]:
# Classes do qiskit
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister, transpile
from qiskit_aer import AerSimulator, Aer
from qiskit.quantum_info import Statevector
from qiskit.visualization import plot_histogram, array_to_latex

import numpy as np

In [22]:
def _syndrome_only_circuit(
    initial_state: str,
    inject_error: int | None,
    round_tag: int,
) -> QuantumCircuit:
    """
    Constrói encode → X̄ → erro → medição de estabilizadores.
    Sem correção nem decodificação: as âncoras são medidas e o resultado
    fica em c_syn (ClassicalRegister nomeado com round_tag para evitar
    colisão ao encadear circuitos).
    """
    q_codes   = QuantumRegister(3, "code")
    q_ancilla = QuantumRegister(2, "ancilla")
    c_syn     = ClassicalRegister(2, f"syn_{round_tag}")

    qc = QuantumCircuit(q_codes, q_ancilla, c_syn)

    # — Encode —
    if initial_state == "1":
        qc.x(q_codes[0])
    qc.cx(q_codes[0], q_codes[1])
    qc.cx(q_codes[0], q_codes[2])

    # — X̄ transversal —
    for i in range(3):
        qc.x(q_codes[i])

    # — Erro —
    if inject_error is not None:
        qc.x(q_codes[inject_error])

    # — S1 = Z₀Z₁ —
    qc.h(q_ancilla[0])
    qc.cz(q_ancilla[0], q_codes[0])
    qc.cz(q_ancilla[0], q_codes[1])
    qc.h(q_ancilla[0])
    qc.measure(q_ancilla[0], c_syn[0])

    # — S2 = Z₁Z₂ —
    qc.h(q_ancilla[1])
    qc.cz(q_ancilla[1], q_codes[1])
    qc.cz(q_ancilla[1], q_codes[2])
    qc.h(q_ancilla[1])
    qc.measure(q_ancilla[1], c_syn[1])

    return qc

def _ancilla_readout_noise(p_err: float) -> NoiseModel:
    """
    ReadoutError simétrico com probabilidade p_err de leitura incorreta,
    aplicado nos qubits físicos 3 (ancilla[0]) e 4 (ancilla[1]).

    Matriz P(leitura | estado real):
        [[1-p, p ],    estado real = |0⟩
         [p,  1-p]]   estado real = |1⟩
    """
    nm = NoiseModel()
    re = ReadoutError([[1 - p_err, p_err],
                       [p_err,     1 - p_err]])
    nm.add_readout_error(re, qubits=[3])
    nm.add_readout_error(re, qubits=[4])
    return nm


def _parse_syndrome(bitstring: str) -> tuple[int, int]:
    """
    Converte a bitstring do Qiskit (MSB à esquerda, espaços entre registros)
    para (s0, s1) inteiros.
    Formato da chave: "s1s0" (bit 0 = s0 = ancilla[0], bit 1 = s1 = ancilla[1]).
    """
    bits = bitstring.replace(" ", "")
    s0 = int(bits[-1])   # bit menos significativo → ancilla[0]
    s1 = int(bits[-2])   # bit seguinte           → ancilla[1]
    return s0, s1


def majority_vote_syndrome(syndrome_list: list[tuple[int, int]]) -> int:
    """
    Dado uma lista de (s0, s1) de N rodadas, retorna o inteiro de 2 bits
    com a síndrome votada (bit 0 = s0_votado, bit 1 = s1_votado).

    Votação bit a bit: cada bit é decidido independentemente pela maioria.
    Empate (N par com N/2 votos) → voto conservador: 0 (sem erro nesse bit).
    """
    n = len(syndrome_list)
    votes_s0 = sum(s0 for s0, _  in syndrome_list)
    votes_s1 = sum(s1 for _,  s1 in syndrome_list)
    s0_voted = 1 if votes_s0 > n / 2 else 0
    s1_voted = 1 if votes_s1 > n / 2 else 0
    return s0_voted | (s1_voted << 1)

def extract_syndromes_noisy(
    initial_state: str,
    inject_error: int | None,
    num_rounds: int,
    p_err: float,
    shots: int,
    sim: AerSimulator,
) -> tuple[dict[int, int], list[dict[int, int]]]:
    """
    Executa `num_rounds` circuitos independentes de extração de síndrome,
    cada um com ruído de readout nas âncoras.

    Para cada shot, reúne os resultados das N rodadas e aplica votação
    majoritária bit a bit, produzindo a síndrome votada.

    Retorna
    -------
    voted_counts    : {síndrome_int: nº de shots} após votação
    per_round_raw   : lista de {síndrome_int: nº de shots} por rodada
    """
    noise_model = _ancilla_readout_noise(p_err)

    # ── Passo A: executar cada rodada e expandir counts em listas ──────────
    # Cada elemento é uma lista de (s0, s1), uma por shot
    rounds_per_shot: list[list[tuple[int, int]]] = []
    per_round_raw: list[dict[int, int]] = []

    for r in range(num_rounds):
        qc  = _syndrome_only_circuit(initial_state, inject_error, round_tag=r)
        qc_t = transpile(qc, sim)
        counts = sim.run(qc_t, noise_model=noise_model, shots=shots, seed_simulator=42 + r).result().get_counts()

        # Contador por síndrome (para relatório)
        raw: dict[int, int] = collections.Counter()
        expanded: list[tuple[int, int]] = []
        for bitstr, freq in counts.items():
            s0, s1 = _parse_syndrome(bitstr)
            syn_int = s0 | (s1 << 1)
            raw[syn_int] += freq
            expanded.extend([(s0, s1)] * freq)

        per_round_raw.append(dict(raw))
        rounds_per_shot.append(expanded)

    # ── Passo B: votação majoritária shot a shot ────────────────────────────
    # Alinhar ao menor número de shots (pode variar por sampling)
    min_shots = min(len(r) for r in rounds_per_shot)
    rounds_per_shot = [r[:min_shots] for r in rounds_per_shot]

    voted_counts: dict[int, int] = collections.Counter()

    for shot_idx in range(min_shots):
        syndromes_this_shot = [rounds_per_shot[r][shot_idx] for r in range(num_rounds)]
        voted = majority_vote_syndrome(syndromes_this_shot)
        voted_counts[voted] += 1

    return dict(voted_counts), per_round_raw


def _correction_decode_from_voted(
    initial_state: str,
    inject_error: int | None,
    voted_syndrome: int,
    shots: int,
    sim: AerSimulator,
) -> dict[str, int]:
    """
    A síndrome já foi determinada pela votação majoritária; a correção
    é aplicada diretamente sem if_test, garantindo que não há ambiguidade.
    """
    CORRECTION_MAP = {1: 0, 3: 1, 2: 2}   # síndrome_int → qubit a corrigir

    q_codes    = QuantumRegister(3, "code")
    logical_bit = ClassicalRegister(1, "logical bit")
    qc = QuantumCircuit(q_codes, logical_bit)

    # Encode
    if initial_state == "1":
        qc.x(q_codes[0])
    qc.cx(q_codes[0], q_codes[1])
    qc.cx(q_codes[0], q_codes[2])

    # X̄ transversal
    for i in range(3):
        qc.x(q_codes[i])

    # Erro
    if inject_error is not None:
        qc.x(q_codes[inject_error])

    # Correção determinística baseada na síndrome votada
    qc.barrier(label="Correction(voted)")
    if voted_syndrome in CORRECTION_MAP:
        qc.x(q_codes[CORRECTION_MAP[voted_syndrome]])

    # Decodificação
    qc.barrier(label="Decoding")
    qc.cx(q_codes[0], q_codes[1])
    qc.cx(q_codes[0], q_codes[2])
    qc.x(q_codes[0])

    # Medição final
    qc.barrier(label="Measure")
    qc.measure(q_codes[0], logical_bit[0])

    qc_t = transpile(qc, sim)
    return sim.run(qc_t, shots=shots).result().get_counts()


# ══════════════════════════════════════════════════════════════════════════════
# 6.  PIPELINE COMPLETO
# ══════════════════════════════════════════════════════════════════════════════

# Rótulos legíveis para cada síndrome
SYNDROME_LABEL = {
    0: "00 → sem erro detectado",
    1: "01 → erro em code[0]",
    2: "10 → erro em code[2]",
    3: "11 → erro em code[1]",
}

def run_noisy_syndrome_pipeline(
    initial_state = "0",
    inject_error: int | None = 0,
    num_rounds: int = 5,
    p_err: float = 0.15,
    shots: int = 1024,
    seed: int = 42,
) -> dict:
    """
    Pipeline completo:
      1. Extrai síndrome em `num_rounds` rodadas com ruído p_err nas âncoras.
      2. Aplica votação majoritária por shot para inferir a síndrome.
      3. Executa correção + decodificação baseada na síndrome votada.
      4. Retorna estatísticas completas.
    """
    sim = AerSimulator(seed_simulator=seed)

    # Etapa 1 & 2: extração ruidosa + votação
    voted_counts, per_round_raw = extract_syndromes_noisy(
        initial_state, inject_error, num_rounds, p_err, shots, sim
    )

    # Síndrome mais votada (entre todos os shots)
    most_voted_syndrome = max(voted_counts, key=voted_counts.get)
    total_voted = sum(voted_counts.values())

    # Etapa 3: correção + decodificação
    final_counts = _correction_decode_from_voted(
        initial_state, inject_error, most_voted_syndrome, shots, sim
    )

    # Taxa de sucesso: estado correto após todo o pipeline
    # Formato da chave: "bit_logico" (único registro nesse circuito)
    total_final = sum(final_counts.values())
    hits = sum(v for k, v in final_counts.items() if k.strip() == initial_state)
    success_rate = hits / total_final if total_final else 0.0

    return {
        "config": {
            "initial_state": initial_state,
            "inject_error":  inject_error,
            "num_rounds":    num_rounds,
            "p_err":         p_err,
            "shots":         shots,
        },
        "per_round_raw":         per_round_raw,
        "voted_counts":          voted_counts,
        "most_voted_syndrome":   most_voted_syndrome,
        "syndrome_label":        SYNDROME_LABEL[most_voted_syndrome],
        "final_counts":          final_counts,
        "success_rate":          success_rate,
    }


# ══════════════════════════════════════════════════════════════════════════════
# 7.  RELATÓRIO  +  ANÁLISE COMPARATIVA
# ══════════════════════════════════════════════════════════════════════════════

def print_report(res: dict) -> None:
    cfg = res["config"]
    W   = 66
    sep = "─" * W

    print("=" * W)
    print("  CÓDIGO DE REPETIÇÃO BIT-FLIP — SÍNDROME RUIDOSA + VOTAÇÃO")
    print("=" * W)
    inj = (f"qubit {cfg['inject_error']} (X bit-flip)"
           if cfg["inject_error"] is not None else "nenhum")
    print(f"  Estado inicial : |{cfg['initial_state']}⟩")
    print(f"  Erro injetado  : {inj}")
    print(f"  Rodadas        : {cfg['num_rounds']}")
    print(f"  p(erro leitura): {cfg['p_err']:.0%}  (ReadoutError nas âncoras)")
    print(f"  Shots/rodada   : {cfg['shots']}")

    # # ── Por rodada ──────────────────────────────────────────────────────────
    # print(f"\n{sep}")
    # print("  SÍNDROMES BRUTAS POR RODADA")
    # print(f"{sep}")
    # for r, raw in enumerate(res["per_round_raw"]):
    #     total = sum(raw.values())
    #     parts = []
    #     for s in range(4):
    #         cnt = raw.get(s, 0)
    #         parts.append(f"S{s:02b}={cnt/total:4.0%}")
    #     print(f"  Rodada {r+1}: " + "  ".join(parts))

    # ── Votação ──────────────────────────────────────────────────────────────
    total_v = sum(res["voted_counts"].values())
    print(f"\n{sep}")
    print("  SÍNDROME APÓS VOTAÇÃO MAJORITÁRIA (bit a bit, por shot)")
    print(f"{sep}")
    for s in range(4):
        cnt    = res["voted_counts"].get(s, 0)
        bar    = "█" * int(cnt / total_v * 30)
        marker = " ◄ inferida" if s == res["most_voted_syndrome"] else ""
        print(f"  [{s:02b}] {SYNDROME_LABEL[s]:30s}  {cnt:5d} ({cnt/total_v:5.1%}) {bar}{marker}")

    print(f"\n  → Diagnóstico: {res['syndrome_label']}")

    # # ── Resultado lógico ────────────────────────────────────────────────────
    # print(f"\n{sep}")
    # print("  RESULTADO LÓGICO FINAL (após correção + decodificação)")
    # print(f"{sep}")
    # for k, v in sorted(res["final_counts"].items()):
    #     print(f"  |{k}⟩ : {v:5d} shots")
    print(f"\n  ✓ Taxa de sucesso lógico: {res['success_rate']:.2%}")
    print("=" * W)


def comparative_analysis(shots: int = 1024) -> None:
    """
    Tabela: sucesso lógico vs p_err e num_rounds.
    Demonstra quantitativamente o ganho da votação majoritária.
    """
    from scipy.stats import binom as _binom

    rounds_list = [1,    3,    5,    10]
    p_list      = [0.0, 0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.35]
    W = 72

    # Ganho teórico (análise binomial)
    print("\n" + "─" * W)
    print("  GANHO TEÓRICO — P(síndrome correta) por bit  [análise binomial]")
    print("─" * W)
    print(f"  {'p_err':>5} | {'1 rodada':>10} | {'5 rodadas':>10} | {'10 rodadas':>10} | {'Ganho(5R)':>10}")
    print("─" * W)
    for p in p_list:
        p1 = 1 - p
        def pmaj(n):
            return sum(_binom.pmf(k, n, p1) for k in range(n//2+1, n+1))
        print(f"  {p:>5.0%} | {p1:>10.3f} | {pmaj(5):>10.3f} | {pmaj(9):>10.3f} | {pmaj(5)-p1:>+10.3f}")
    print("=" * W)


# ══════════════════════════════════════════════════════════════════════════════
# MAIN
# ══════════════════════════════════════════════════════════════════════════════

if __name__ == "__main__":
  for p_err in [0.0, 0.05, 0.10, 0.20, 0.30]:
      for r in [1, 5, 10]:
          res = run_noisy_syndrome_pipeline(
              initial_state='0',
              inject_error=0,
              num_rounds=r,
              p_err=p_err,
              shots=1024
          )
          print_report(res)
  comparative_analysis(shots=1024)

  CÓDIGO DE REPETIÇÃO BIT-FLIP — SÍNDROME RUIDOSA + VOTAÇÃO
  Estado inicial : |0⟩
  Erro injetado  : qubit 0 (X bit-flip)
  Rodadas        : 1
  p(erro leitura): 0%  (ReadoutError nas âncoras)
  Shots/rodada   : 1024

──────────────────────────────────────────────────────────────────
  SÍNDROME APÓS VOTAÇÃO MAJORITÁRIA (bit a bit, por shot)
──────────────────────────────────────────────────────────────────
  [00] 00 → sem erro detectado             0 ( 0.0%) 
  [01] 01 → erro em code[0]             1024 (100.0%) ██████████████████████████████ ◄ inferida
  [10] 10 → erro em code[2]                0 ( 0.0%) 
  [11] 11 → erro em code[1]                0 ( 0.0%) 

  → Diagnóstico: 01 → erro em code[0]

  ✓ Taxa de sucesso lógico: 100.00%
  CÓDIGO DE REPETIÇÃO BIT-FLIP — SÍNDROME RUIDOSA + VOTAÇÃO
  Estado inicial : |0⟩
  Erro injetado  : qubit 0 (X bit-flip)
  Rodadas        : 5
  p(erro leitura): 0%  (ReadoutError nas âncoras)
  Shots/rodada   : 1024

────────────────────────────────────